# Can an agent select and validate forecasting models better than a fixed policy? And does its reasoning hold up?

**Research question (RQ):** *Can an LLM agent select and validate forecasting models better than a fixed policy, and does its reasoning trace hold up under scrutiny?*

**What this notebook does (Walmart data only, no extra datasets).** We treat each of the **45 stores as its own time series**. For every store the agent:
1. **Profiles** the series: frequency, missing weeks, stationarity, seasonality, outliers, holiday effect.
2. **Proposes candidate models with a justification** (rules by default; an LLM may propose, code validates).
3. **Fits and backtests** the candidates on past folds only.
4. **Selects** one model per store and writes an **analyst memo** with assumptions and caveats.

Then we test the two halves of the RQ:
| Part of the RQ | How we test it | Evidence you get |
|---|---|---|
| *Better than a fixed policy?* | Compare the agent's per-store choice with **fixed policies** ("always ETS", "always LightGBM", "best single model for the chain") and with an **oracle** (best model in hindsight). Selection uses only past folds; scoring uses the next fold. | Error table and *regret* (how far each policy is from the oracle) |
| *Does the reasoning trace hold up?* | Every claim in the agent's trace is **recomputed from the data** by a separate checker. Then we **inject wrong claims** and see how many the checker catches. | % claims verified, % injected faults caught |

**Honest limits (say these in the review):**
- All 45 stores are the same retailer with the same dates, so they are **similar series**; the profile will look alike for most stores, and a selector has little room to shine. Other datasets are future scope.
- Offline, the trace is written by code, so it is correct by construction. The audit proves its value on **injected errors** and is ready to check an LLM-written trace.
- We never claim the agent "wins" unless the numbers show it, and we test whether any win is statistically real.

**To run:** keep `Walmart.csv` in the same folder. About 3-4 minutes. Optional: `GEMINI_API_KEY` for the LLM parts (everything runs without it).

In [4]:
%pip install -q lightgbm statsmodels scipy

Note: you may need to restart the kernel to use updated packages.


In [67]:
import os, getpass
os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key and press Enter: ")
print("key set:", bool(os.environ["GEMINI_API_KEY"]))

Paste your Gemini API key and press Enter:  ········


key set: True


In [68]:
import os
print("key set:", bool(os.environ.get("GEMINI_API_KEY")))

key set: True


In [59]:
import os, getpass

# ---- API key: set ONCE here. Every LLM call below reads it through get_client() / ask_llm(). ----
if not os.environ.get("GEMINI_API_KEY", "").strip():
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key and press Enter: ")
os.environ["GEMINI_API_KEY"] = os.environ["GEMINI_API_KEY"].strip().strip("'\"")      # remove stray spaces/quotes from pasting
GEMINI_MODEL = "gemini-3.5-flash-lite"

def get_client():
    """Return a Gemini client built from the key set above, or None if no key is set."""
    key = os.environ.get("GEMINI_API_KEY", "").strip()
    if not key: return None
    from google import genai
    return genai.Client(api_key=key)

def ask_llm(prompt):
    """Send one prompt to Gemini (temperature 0) and return the text, or None if no key is set."""
    client = get_client()
    if client is None: return None
    from google.genai import types
    return client.models.generate_content(model=GEMINI_MODEL, contents=prompt,
                                          config=types.GenerateContentConfig(temperature=0)).text

print("key set:", bool(os.environ.get("GEMINI_API_KEY")))


key set: True


In [38]:
import os
print("key set:", bool(os.environ.get("GEMINI_API_KEY")))

key set: True


## 0. Setup: data, features, models and the rolling-origin backtest (same engine as your ML notebook)
**Rolling origin:** 4 times, train on weeks before the origin, forecast the next 12 weeks. Folds 1-4 are used below as an *expanding test*: to judge a selection we pick models using only earlier folds and score on the next fold. No fold is ever used to choose a model that is then scored on the same fold.

In [39]:
import os, re, json, warnings                          # os/json/re: key, LLM answers, text patterns; warnings: hide noise
warnings.filterwarnings("ignore")                      # clean output for beginners
import numpy as np, pandas as pd                       # maths and tables
from scipy import stats                                # Wilcoxon test
from lightgbm import LGBMRegressor                     # global gradient-boosting model
from statsmodels.tsa.holtwinters import ExponentialSmoothing   # ETS
from statsmodels.tsa.seasonal import STL               # seasonal decomposition (for the profile)
from statsmodels.tsa.stattools import adfuller         # stationarity test (for the profile)
try: display
except NameError: display = print                      # plain-script fallback

H, ORIGINS = 12, [95, 107, 119, 131]                   # horizon and the 4 rolling origins
df = pd.read_csv(r"C:\Users\kumar\Desktop\Study\VIT\Mona - VIT\Semester 4\Project\walmart-sales-dataset-of-45stores.csv");
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")   # read data, real dates
df = df.sort_values(["Store", "Date"]).reset_index(drop=True)   # each store in time order
dates = sorted(df.Date.unique())                       # the 143 distinct weeks
print(df.shape, "| stores:", df.Store.nunique(), "| weeks:", len(dates))

(6435, 8) | stores: 45 | weeks: 143


In [40]:
def build_features(t):
    f = t.copy(); g = f.groupby("Store")["Weekly_Sales"]                                   # per-store sales in time order
    f["lag12"], f["lag26"], f["lag52"] = g.shift(12), g.shift(26), g.shift(52)             # sales 12/26/52 weeks earlier (never newer: 12-week-ahead forecast)
    f["roll13"] = g.transform(lambda s: s.shift(12).rolling(13).mean())                    # 13-week average ending 12 weeks ago
    f["f12"], f["f26"], f["froll"] = np.log(f.lag12 / f.lag52), np.log(f.lag26 / f.lag52), np.log(f.roll13 / f.lag52)   # ratios to last year
    f["week"], f["month"] = f.Date.dt.isocalendar().week.astype(int), f.Date.dt.month      # calendar features
    f["sin52"], f["cos52"] = np.sin(2 * np.pi * f.week / 52), np.cos(2 * np.pi * f.week / 52)   # smooth yearly cycle
    fl = f.Holiday_Flag == 1                                                               # official holiday weeks
    for n, m in [("ev_superbowl", 2), ("ev_laborday", 9), ("ev_thanks", 11), ("ev_xmas", 12)]: f[n] = (fl & (f.month == m)).astype(int)   # holiday type
    f["ev_preXmas"] = ((f.month == 12) & f.Date.dt.day.between(20, 26) & ~fl).astype(int)  # week before Christmas (flag misses it)
    return f

flag_by_date = df.groupby("Date")["Holiday_Flag"].max()                                    # holiday flag per date
full = df[["Store", "Date", "Weekly_Sales", "Holiday_Flag"]]                               # only history is needed here (no future weeks in this experiment)
feat = build_features(full)                                                                # feature table
FEATS = ["week", "month", "sin52", "cos52", "Holiday_Flag", "ev_superbowl", "ev_laborday", "ev_thanks", "ev_xmas", "ev_preXmas", "f12", "f26", "froll"]

# leakage test: hide sales after fold 1's origin, rebuild features, forecast-week features must be identical
O = ORIGINS[0]; hid = full.copy(); hid.loc[hid.Date >= dates[O], "Weekly_Sales"] = np.nan
a = feat[feat.Date.isin(dates[O:O + H])][FEATS + ["lag52"]]; b = build_features(hid).loc[a.index, FEATS + ["lag52"]]
assert np.allclose(a.fillna(-1).values, b.fillna(-1).values), "LEAK: features use future sales"
print("Leakage test passed")

Leakage test passed


In [11]:
%pip install -U --force-reinstall lightgbm

  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
  Using cached narwhals-2.27.1-py3-none-any.whl.metadata (15 kB)
  Using cached numpy-2.5.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl.metadata (61 kB)
Using cached lightgbm-4.7.0-py3-none-win_amd64.whl (1.4 MB)
Using cached narwhals-2.27.1-py3-none-any.whl (483 kB)
Using cached numpy-2.5.4-cp313-cp313-win_amd64.whl (12.6 MB)
Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl (36.6 MB)

  Attempting uninstall: numpy

    Found existing installation: numpy 2.5.4

   ---------------------------------------- 0/4 [numpy]
   ---------------------------------------- 0/4 [numpy]
    Uninstalling numpy-2.5.4:
   ---------------------------------------- 0/4 [numpy]
      Successfully uninstalled numpy-2.5.4
   ---------------------------------------- 0/4 [numpy]
   ---------------------------------------- 0/4 [numpy]
   ---------------------------------------- 0/4 [numpy

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


In [41]:
def seasonal_naive(train, test): return test["lag52"].values                              # same week last year

def ets(train, test):                                                                      # one ETS model per store on log sales
    pred = pd.Series(np.nan, index=test.index)
    for s, tr in train.groupby("Store"):
        idx = test.index[test.Store == s]
        if len(tr) >= 104: pred[idx] = np.exp(ExponentialSmoothing(np.log(tr.Weekly_Sales.values), trend="add", damped_trend=True, seasonal="add", seasonal_periods=52).fit().forecast(len(idx)))
        else: pred[idx] = test.loc[idx, "lag52"].values                                    # too short: fall back to seasonal naive
    return pred.values

def lightgbm(train, test):                                          # pooled gradient boosting (scikit-learn version)
    from sklearn.ensemble import HistGradientBoostingRegressor
    tr = train.dropna(subset=FEATS + ["lag52"])                     # rows where every feature exists
    X = tr[FEATS].to_numpy(dtype="float64")                         # features as plain NumPy
    y = np.log(tr.Weekly_Sales / tr.lag52).to_numpy(dtype="float64")   # target = log ratio to last year
    m = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, max_leaf_nodes=15, random_state=42)
    m.fit(X, y, sample_weight=np.where(tr.Holiday_Flag == 1, 5.0, 1.0))   # holiday weeks count 5x
    return test["lag52"].to_numpy(dtype="float64") * np.exp(m.predict(test[FEATS].to_numpy(dtype="float64")))

MODELS = ["SeasonalNaive", "ETS", "LightGBM", "Hybrid"]                                    # simplest -> most complex (used for tie-breaks)
def backtest():
    rows = []
    for k, O in enumerate(ORIGINS, 1):                                                     # 4 folds
        train = feat[(feat.Date < dates[O]) & feat.Weekly_Sales.notna()]; test = feat[feat.Date.isin(dates[O:O + H])]
        P = {"SeasonalNaive": seasonal_naive(train, test), "ETS": ets(train, test), "LightGBM": lightgbm(train, test)}
        P["Hybrid"] = 0.5 * P["ETS"] + 0.5 * P["LightGBM"]                                 # average of the two
        for n, p in P.items(): rows.append(test[["Store", "Date", "Holiday_Flag"]].assign(y=test.Weekly_Sales.values, model=n, yhat=p, fold=k))
    r = pd.concat(rows); r["abs_err"] = (r.y - r.yhat).abs(); r["w"] = np.where(r.Holiday_Flag == 1, 5, 1); r["wabs"] = r.w * r.abs_err   # weights for WMAE
    return r
cv = backtest()                                                                            # all stores x models x folds
ND = cv.groupby(["Store", "model", "fold"]).agg(N=("wabs", "sum"), D=("w", "sum"))        # numerator/denominator of WMAE per store-model-fold
ND["wmae"] = ND.N / ND.D                                                                   # per-store WMAE (lower is better)
print("backtest rows:", len(cv), "| models:", MODELS)

backtest rows: 8640 | models: ['SeasonalNaive', 'ETS', 'LightGBM', 'Hybrid']


## 1. Profile agent: what kind of series is this?
**Six measurements per store** (computed only on weeks **before the last test fold**, so the profile never peeks at the final forecast window):
| Check | How | Meaning |
|---|---|---|
| Frequency | median gap between dates | weekly = 7 days |
| Missingness | expected weeks minus weeks present, plus blank values | gaps break seasonal models |
| Stationarity | ADF test on log sales (p < 0.05 -> stationary) | trending series need trend models |
| Seasonality strength | correlation of each week with the same week a year earlier, after removing the trend (0 = none, 1 = identical every year) | strong -> use seasonal models |
| Outliers | share of weeks with a robust z-score > 3.5 on the STL residual | spikes need holiday/event handling |
| Holiday effect | mean sales in holiday weeks / other weeks | big effect -> use holiday features |

**Quality check first:** four tiny synthetic series with a known answer test the profile code itself.

In [42]:
def profile_series(dt, y):
    """dt: dates, y: sales values of ONE series. Returns the six measurements as a dict."""
    y = pd.Series(np.asarray(y, dtype="float64"), index=pd.to_datetime(dt))              # series with a date index
    gap = int(np.median(np.diff(y.index.values).astype("timedelta64[D]").astype(int)))   # median days between observations
    expected = int((y.index.max() - y.index.min()).days / gap) + 1                        # weeks that should exist
    missing = (expected - len(y)) + int(y.isna().sum())                                   # absent weeks + blank values
    ly = np.log(y.dropna())                                                               # log scale
    adf_p = float(adfuller(ly, maxlag=4, autolag=None)[1])                                # ADF p-value with 4 lags (small = stationary)
    if len(ly) >= 104:                                                                    # need two full years to compare year with year
        x = np.arange(len(ly)); d = ly.values - np.polyval(np.polyfit(x, ly.values, 1), x)   # log sales with the straight-line trend removed
        seas = max(0.0, float(np.corrcoef(d[52:], d[:-52])[0, 1]))                        # correlation of each week with the same week a year earlier
        r = STL(ly, period=52).fit().resid                                                # what is left after trend and season (for outliers)
    else: r, seas = ly - ly.rolling(8, min_periods=1).median(), None                      # too short: no seasonality estimate
    z = 0.6745 * (r - r.median()) / (np.median(np.abs(r - r.median())) + 1e-12)           # robust z-score of the residual
    return {"frequency_days": gap, "missing": missing, "adf_p": round(adf_p, 3), "seasonal_strength": None if seas is None else round(float(seas), 2),
            "outlier_share": round(float((np.abs(z) > 3.5).mean()), 3), "n_weeks": int(len(y))}

def holiday_effect(g): return round(float(g[g.Holiday_Flag == 1].Weekly_Sales.mean() / g[g.Holiday_Flag == 0].Weekly_Sales.mean()), 2)   # holiday / normal-week sales

# unit tests on synthetic series with a KNOWN answer
t = pd.date_range("2010-01-01", periods=140, freq="7D"); k = np.arange(140)
sine = 1000 * np.exp(0.3 * np.sin(2 * np.pi * k / 52))                                    # strong yearly cycle
noise = 1000 * np.exp(np.random.default_rng(0).normal(0, 0.1, 140))                       # no cycle, stationary
walk = 1000 * np.exp(np.cumsum(np.random.default_rng(1).normal(0, 0.05, 140)))            # random walk = non-stationary
spiky = noise.copy(); spiky[70] *= 6                                                       # one big spike
gappy = pd.Series(noise, index=t).drop(t[[30, 31, 32]])                                    # 3 weeks removed
assert profile_series(t, sine)["seasonal_strength"] > 0.8, "sine should be strongly seasonal"
assert profile_series(t, noise)["seasonal_strength"] < 0.3 and profile_series(t, noise)["adf_p"] < 0.05, "noise: weak seasonality, stationary"
assert profile_series(t, walk)["adf_p"] > 0.05, "random walk should be non-stationary"
assert profile_series(t, spiky)["outlier_share"] > profile_series(t, noise)["outlier_share"], "spike should raise outlier share"
assert profile_series(gappy.index, gappy.values)["missing"] == 3, "should detect 3 missing weeks"
print("5 profile unit tests passed")

5 profile unit tests passed


In [43]:
CUT = dates[ORIGINS[-1]]                                                                   # profile only weeks before the last fold's origin
prof = {}                                                                                  # store -> profile dict
for s, g in df[df.Date < CUT].groupby("Store"):                                            # one profile per store
    prof[s] = {**profile_series(g.Date, g.Weekly_Sales), "holiday_effect": holiday_effect(g)}
P = pd.DataFrame(prof).T.astype(float); P.index.name = "Store"                            # table: one row per store
display(P.describe().loc[["min", "50%", "max"]].round(2))                                  # spread across stores
print("Stores that look stationary (ADF p<0.05):", int((P.adf_p < 0.05).sum()), "of 45 | strong seasonality (>=0.5):", int((P.seasonal_strength >= 0.5).sum()), "of 45")

,frequency_days,missing,adf_p,seasonal_strength,outlier_share,n_weeks,holiday_effect
min,7.0,0.0,0.00,0.25,0.07,131.0,0.97
50%,7.0,0.0,0.00,0.86,0.16,131.0,1.08
max,7.0,0.0,0.95,0.94,0.37,131.0,1.21


Stores that look stationary (ADF p<0.05): 38 of 45 | strong seasonality (>=0.5): 41 of 45


### What to notice
Most Walmart stores share almost the same profile (weekly, no gaps, strong seasonality, a few holiday outliers), so **profile-based rules propose the same candidates for most stores** (in our run, 41 of 45 get all four models; the 4 stores with weak yearly seasonality lose ETS and Hybrid). The agent can still pick *different winners per store* because the backtest decides. This is a limit of using only Walmart, and a reason to list "more heterogeneous datasets" as future scope.

## 2. Candidate proposal with justification, then validation and selection
**Candidate rules** (each produces a *justification claim* that the audit will check later):
- **SeasonalNaive:** always included (the baseline everything must beat).
- **ETS:** included if the store has >= 104 weeks **and** seasonal strength >= 0.5.
- **LightGBM:** included if the chain has >= 10 stores and the store has >= 60 weeks (it is pooled across stores).
- **Hybrid:** included when both ETS and LightGBM are included.

**Selection rule (decided before looking at results):** for each store, use only folds available *before* the test fold; take the model with the lowest WMAE; if a simpler model is within **3%** of the best, take the simpler one (parsimony: do not pay for complexity that is not proven). We also report the same agent **without** the 3% rule, to show whether that choice matters.

**Where the LLM can help (optional):** it may *suggest* candidates from the menu; code accepts only names on the menu and never lets the LLM choose the winner.

In [45]:
import traceback
print("key set:", bool(os.environ.get("GEMINI_API_KEY")), "| key length:", len(os.environ.get("GEMINI_API_KEY", "")))
try:
    import google.genai as g; print("google-genai version:", getattr(g, "__version__", "unknown"))
    print(ask_llm("Reply with the single word: ok"))
except Exception:
    traceback.print_exc()


key set: True | key length: 53
google-genai version: 2.31.0


Traceback (most recent call last):
  File "C:\Users\kumar\AppData\Local\Temp\ipykernel_21636\1674922226.py", line 5, in <module>
    print(ask_llm("Reply with the single word: ok"))
          ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\kumar\AppData\Local\Temp\ipykernel_21636\728478992.py", line 21, in ask_llm
    return client.models.generate_content(model=GEMINI_MODEL, contents=prompt,
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                                          config=types.GenerateContentConfig(temperature=0)).text
                                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kumar\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\genai\models.py", line 6288, in generate_content
    response = self._generate_content_with_continuation(
        model=model, contents=contents, config=parsed_config_to_call
    )
  File "c:\Users\kumar\AppData\Local\Programs\Python\Py

In [65]:
MENU = {"SeasonalNaive": "same week last year", "ETS": "per-store exponential smoothing", "LightGBM": "pooled gradient boosting", "Hybrid": "average of ETS and LightGBM"}
TOL = 0.03                                                                                 # parsimony tolerance

def rule_candidates(p, n_stores=45):
    """Return (candidate list, list of justification claims) for one store profile."""
    c, claims = ["SeasonalNaive"], []
    claims.append({"kind": "profile", "metric": "n_weeks", "op": ">=", "thr": 104, "value": p["n_weeks"], "model": "ETS"})            # ETS needs 2 years
    claims.append({"kind": "profile", "metric": "seasonal_strength", "op": ">=", "thr": 0.5, "value": p["seasonal_strength"], "model": "ETS"})   # ETS needs seasonality
    if p["n_weeks"] >= 104 and (p["seasonal_strength"] or 0) >= 0.5: c.append("ETS")
    claims.append({"kind": "profile", "metric": "n_weeks", "op": ">=", "thr": 60, "value": p["n_weeks"], "model": "LightGBM"})        # LightGBM needs history
    if n_stores >= 10 and p["n_weeks"] >= 60: c.append("LightGBM")
    if "ETS" in c and "LightGBM" in c: c.append("Hybrid")
    return c, claims

LLM_CACHE = {}
def llm_candidates(p):
    """Optional: LLM suggests candidates from the menu. Returns a validated list or None."""
    if get_client() is None: return None                                                   # no key / quota used up -> rules are used
    ck = (int(p["n_weeks"]), round(p["seasonal_strength"] or 0, 1), p["adf_p"] < 0.05, round(p["outlier_share"], 2), round(p.get("holiday_effect", 0), 1))
    if ck in LLM_CACHE: return LLM_CACHE[ck]                                              # near-identical profile already asked: reuse (saves free-tier calls)                                                   # no key -> rules are used
    try:
        r = ask_llm(f"Series profile: {p}. Menu: {MENU}. Return only a JSON list of model names worth testing.")
        if r is None: return None                                                           # LLM unavailable (quota) -> rules
        names = [n for n in json.loads(re.search(r"\[.*\]", r, re.S).group()) if n in MENU]   # accept only names on the menu
        out = sorted(set(names) | {"SeasonalNaive"}, key=MODELS.index) if names else None; LLM_CACHE[ck] = out; return out
    except Exception as e: print("LLM failed, using rules:", type(e).__name__, "-", str(e)[:200]); return None

def select(store, cands, folds, tol=TOL):
    """Pick one model for `store` using only the given earlier folds. Returns (model, mean WMAE per candidate)."""
    w = {m: float(np.mean([ND.loc[(store, m, k), "wmae"] for k in folds])) for m in cands}   # validation WMAE per candidate
    best = min(w, key=w.get)                                                               # lowest error
    simple = [m for m in MODELS if m in w and w[m] <= w[best] * (1 + tol)]                 # models within the tolerance of the best
    return (simple[0] if tol > 0 else best), w                                             # simplest of those (or the best if tol=0)

## 3. The experiment: agent vs fixed policies vs oracle
For each test fold **k = 2, 3, 4**: every policy chooses using folds **before k** only, then is scored on fold k.
- **Fixed policies:** always SeasonalNaive / ETS / LightGBM / Hybrid.
- **Best single model:** the one model with the lowest chain-wide validation error on earlier folds (what your ML notebook does).
- **Agent:** per-store selection with the rules above (with and without the 3% parsimony rule).
- **Oracle:** the best model for each store *in hindsight* on fold k. Nobody can achieve it; it is the floor that measures **regret** = how much worse than perfect.

Score = chain WMAE on fold k (holiday weeks count 5x), averaged over k = 2, 3, 4. Lower is better.

In [50]:
cands, claims = rule_candidates(prof[1])        # prof comes from the profile section
print("Store 1 candidates:", cands)
print("Selected model:", select(1, cands, [1, 2, 3]))

Store 1 candidates: ['SeasonalNaive', 'ETS', 'LightGBM', 'Hybrid']
Selected model: ('ETS', {'SeasonalNaive': 94042.3722222222, 'ETS': 68962.61909438416, 'LightGBM': 73762.99748274141, 'Hybrid': 68097.99027092507})


In [60]:
print(llm_candidates(prof[1]))

['SeasonalNaive', 'LightGBM', 'Hybrid']


In [69]:
stores = sorted(df.Store.unique())
def chain_wmae(choice, k):                                                                 # choice: Series store -> model
    N = sum(ND.loc[(s, choice[s], k), "N"] for s in stores); D = sum(ND.loc[(s, choice[s], k), "D"] for s in stores); return N / D

cands_by_store = {s: rule_candidates(prof[s])[0] for s in stores}                          # rule-based candidate lists
llm_used = False
for s in stores:                                                                           # optional LLM override (validated)
    l = llm_candidates(prof[s]) if os.environ.get("GEMINI_API_KEY") else None
    if l: cands_by_store[s], llm_used = l, True
print("Candidate lists:", pd.Series({s: tuple(c) for s, c in cands_by_store.items()}).value_counts().to_dict(), "| by", "LLM (validated)" if llm_used else "rules")

pol_names = ["Always SeasonalNaive", "Always ETS", "Always LightGBM", "Always Hybrid", "Best single model", "Agent (no parsimony)", "Agent (3% parsimony)", "Oracle (hindsight)"]
score, choices = {p: {} for p in pol_names}, {}                                            # policy -> {fold: score}; (policy, fold) -> store->model
for k in (2, 3, 4):
    past = list(range(1, k))                                                               # folds before k
    ch = {"Always " + m: pd.Series(m, index=stores) for m in MODELS}                       # fixed policies
    chain_val = {m: np.mean([ND.xs((m, f), level=("model", "fold")).N.sum() / ND.xs((m, f), level=("model", "fold")).D.sum() for f in past]) for m in MODELS}   # chain error per model on past folds
    ch["Best single model"] = pd.Series(min(chain_val, key=chain_val.get), index=stores)  # one model for every store
    ch["Agent (no parsimony)"] = pd.Series({s: select(s, cands_by_store[s], past, tol=0)[0] for s in stores})   # per-store winner
    ch["Agent (3% parsimony)"] = pd.Series({s: select(s, cands_by_store[s], past, tol=TOL)[0] for s in stores})
    ch["Oracle (hindsight)"] = pd.Series({s: min(MODELS, key=lambda m: ND.loc[(s, m, k), "wmae"]) for s in stores})   # best on the test fold itself
    for p in pol_names: score[p][k] = chain_wmae(ch[p], k); choices[(p, k)] = ch[p]

R = pd.DataFrame(score).T; R["mean WMAE"] = R[[2, 3, 4]].mean(axis=1)                      # per-fold and mean score
orc = R.loc["Oracle (hindsight)", "mean WMAE"]
R["regret vs oracle"] = (100 * (R["mean WMAE"] / orc - 1)).round(1).astype(str) + "%"      # how far from perfect
R.columns = ["fold 2", "fold 3", "fold 4", "mean WMAE", "regret vs oracle"]
display(R.round(0))
print("Best single model chosen per fold:", {k: choices[("Best single model", k)].iloc[0] for k in (2, 3, 4)})
print("Agent (3%) choices on fold 4:", choices[("Agent (3% parsimony)", 4)].value_counts().to_dict())

Candidate lists: {('SeasonalNaive', 'LightGBM', 'Hybrid'): 32, ('SeasonalNaive', 'ETS', 'LightGBM', 'Hybrid'): 13} | by LLM (validated)


,fold 2,fold 3,fold 4,mean WMAE,regret vs oracle
Always SeasonalNaive,66379.0,55019.0,56385.0,59261.0,64.3%
Always ETS,47055.0,39424.0,41366.0,42615.0,18.2%
Always LightGBM,53066.0,49235.0,44037.0,48779.0,35.3%
Always Hybrid,45319.0,39321.0,34487.0,39709.0,10.1%
Best single model,45319.0,39321.0,34487.0,39709.0,10.1%
Agent (no parsimony),49257.0,40091.0,38896.0,42748.0,18.5%
Agent (3% parsimony),50065.0,42364.0,38673.0,43700.0,21.2%
Oracle (hindsight),42755.0,35533.0,29910.0,36066.0,0.0%


Best single model chosen per fold: {2: 'Hybrid', 3: 'Hybrid', 4: 'Hybrid'}
Agent (3%) choices on fold 4: {'Hybrid': 37, 'ETS': 4, 'LightGBM': 3, 'SeasonalNaive': 1}


In [70]:
# is the agent really different from the best single model? paired Wilcoxon on per-store WMAE over folds 2-4
def per_store(policy):
    return pd.Series({s: np.mean([ND.loc[(s, choices[(policy, k)][s], k), "wmae"] for k in (2, 3, 4)]) for s in stores})
a, b = per_store("Agent (3% parsimony)"), per_store("Best single model")
w = stats.wilcoxon(a, b)                                                                   # paired test across 45 stores
print(f"Agent vs Best single model: agent better in {(a < b).mean():.0%} of stores | mean gain {100 * (1 - a.mean() / b.mean()):+.1f}% | Wilcoxon p = {w.pvalue:.3f}")
print("Reading: p < 0.05 would mean a real difference; otherwise report it as 'no clear difference'.")

Agent vs Best single model: agent better in 2% of stores | mean gain -10.1% | Wilcoxon p = 0.000
Reading: p < 0.05 would mean a real difference; otherwise report it as 'no clear difference'.


### How to read the experiment (be honest)
- **Look at the oracle row first.** If even the oracle is only a few percent better than the best single model, per-store selection has little to gain on this data, and no selector can look impressive.
- **If the agent is not clearly better than the best single model, say so.** The result "per-store selection does not beat a good single model on Walmart" is a valid finding. Selection from only 12 weeks of validation per fold can also *overfit* (it picks last period's lucky model).
- **The parsimony rows** show whether the 3% rule helped; do not tune it after seeing the test fold (we fixed it in advance).
- **In my run (check yours):** the per-store agent was *worse* than the best single model (about 9% higher error, better in only about a quarter of stores, Wilcoxon p = 0.001). Choosing per store from 12-week windows over-fits: it follows last period's luck. "Always Hybrid" had the lowest error among the fixed policies, but that is only known in hindsight; at fold 2 the best single model chosen from past data was LightGBM.
- So the honest answer to the first half of the RQ on Walmart is **no, selection per store did not beat a good single model**. Say this plainly, explain why (little profile variation, short validation windows), and present more series and longer validation as future scope.
- The agent's remaining selling point is the **auditable reasoning** tested next, not an accuracy win.

## 4. Reasoning trace and audit ("does it hold up under scrutiny?")
**Trace:** for each store, a list of **claims**: why each model was proposed (profile facts), how candidates scored in the backtest, and why one was chosen. Each claim is stored as structured fields *and* as a sentence.

**Audit (a separate checker that trusts nothing in the trace):**
1. **Recompute the value** from the raw data (profile numbers) or from the backtest table (scores).
2. **Check the logic:** a claim "seasonal strength 0.93 >= 0.50" must really satisfy the inequality, and the stated winner must be the lowest score under the selection rule.
3. **Check the sentence:** the text must equal the sentence generated from the fields (so words cannot drift from the numbers).

**Fault injection:** we corrupt claims on purpose (change a number, flip an inequality, name the wrong winner, change the text) and measure the share the audit catches.

In [74]:
def render(c):                                                                             # the one allowed sentence per claim (numbers come from fields only)
    if c["kind"] == "profile": return f"{c['model']} considered because {c['metric']} = {c['value']} ({c['op']} {c['thr']}) -> {'satisfied' if c['ok'] else 'not satisfied'}."
    if c["kind"] == "backtest": return f"{c['model']} validation WMAE on folds {c['folds']} = {c['value']:,.0f}."
    if c["kind"] == "decision": return f"Selected {c['model']} (best validation WMAE {c['best']:,.0f}; {c['model']} = {c['value']:,.0f}, tolerance {c['tol']:.0%})."

def build_trace(s, folds):
    c, prof_claims = rule_candidates(prof[s]); trace = []
    for pc in prof_claims:                                                                 # why each model was (not) proposed
        pc = {**pc, "ok": bool(pc["value"] is not None and (pc["value"] >= pc["thr"]))}; trace.append({**pc, "text": render(pc)})
    model, w = select(s, cands_by_store[s], folds)                                          # validation scores and the pick
    for m, v in w.items(): bc = {"kind": "backtest", "model": m, "folds": list(folds), "value": round(v, 0)}; trace.append({**bc, "text": render(bc)})
    dc = {"kind": "decision", "model": model, "best": round(min(w.values()), 0), "value": round(w[model], 0), "tol": TOL}; trace.append({**dc, "text": render(dc)})
    return trace

def audit_claim(s, c):
    """Return (ok, reason). Recomputes everything from raw data / backtest table."""
    if c["text"] != render(c): return False, "sentence does not match its fields"                              # check 3
    if c["kind"] == "profile":
        g = df[(df.Store == s) & (df.Date < CUT)]; fresh = profile_series(g.Date, g.Weekly_Sales)               # recompute from raw data
        if c["metric"] not in fresh or fresh[c["metric"]] != c["value"]: return False, f"{c['metric']} should be {fresh.get(c['metric'])}"   # check 1
        if c["ok"] != (c["value"] >= c["thr"]): return False, "verdict contradicts the inequality"               # check 2
        return True, "ok"
    if c["kind"] == "backtest":
        if c["model"] not in MODELS: return False, "unknown model"                                              # model must exist
        true = round(float(np.mean([ND.loc[(s, c["model"], k), "wmae"] for k in c["folds"]])), 0)              # recompute from the backtest table
        return (abs(true - c["value"]) <= 1), ("ok" if abs(true - c["value"]) <= 1 else f"WMAE should be {true:,.0f}")
    return True, "ok"                                                                                          # decisions are checked in audit_trace

def audit_trace(s, trace, folds):
    res = [audit_claim(s, c) for c in trace]                                                                   # claim-level checks
    dec = trace[-1]; model, w = select(s, cands_by_store[s], folds)                                            # re-derive the decision from the data, not from the trace
    if dec["model"] != model: res[-1] = (False, f"decision should be {model}")                                  # winner must follow the rule
    elif dec["best"] != round(min(w.values()), 0) or dec["value"] != round(w[model], 0): res[-1] = (False, "decision numbers do not match backtest")
    return res

folds = [1, 2, 3]                                                                          # selection folds for the final trace (fold 4 stays unseen)
traces = {s: build_trace(s, folds) for s in stores}
ok = [r[0] for s in stores for r in audit_trace(s, traces[s], folds)]
print(f"Clean traces: {sum(ok)}/{len(ok)} claims verified ({100 * np.mean(ok):.0f}%). Offline this is true by construction; the next test shows the audit really detects errors.")

Clean traces: 328/328 claims verified (100%). Offline this is true by construction; the next test shows the audit really detects errors.


In [72]:
rng = np.random.default_rng(0)
def corrupt(c, how):
    c = dict(c)
    if how == "number x1.3" and "value" in c and c["value"] is not None: c["value"] = c["value"] * 1.3; c["text"] = render(c)   # wrong number, consistent text
    elif how == "verdict flipped" and c["kind"] == "profile": c["ok"] = not c["ok"]; c["text"] = render(c)
    elif how == "wrong winner" and c["kind"] == "decision": c["model"] = [m for m in MODELS if m != c["model"]][0]; c["text"] = render(c)
    elif how == "text edited": c["text"] = c["text"].replace("=", "= about") if "=" in c["text"] else c["text"] + " (strongly)"
    else: return None                                                                       # this fault does not apply to this claim type
    return c
res = []
for how in ["number x1.3", "verdict flipped", "wrong winner", "text edited"]:
    caught = total = 0
    for s in stores:
        for i, c in enumerate(traces[s]):
            bad = corrupt(c, how)
            if bad is None: continue
            tr = list(traces[s]); tr[i] = bad                                              # replace one claim with its corrupted version
            r = audit_trace(s, tr, folds)[i]; total += 1; caught += (not r[0])             # caught if the audit rejects that claim
    res.append({"fault": how, "injected": total, "caught": caught, "caught_%": round(100 * caught / max(total, 1))})
display(pd.DataFrame(res))

,fault,injected,caught,caught_%
0,number x1.3,328,328,100
1,verdict flipped,135,135,100
2,wrong winner,45,45,100
3,text edited,328,328,100


### How to explain the audit
- **What it proves:** every number and verdict in the trace can be traced back to the data; any edited number, flipped verdict, wrong winner or altered wording is caught.
- **What it does not prove:** that the *reasoning is wise*. It checks facts and rule-following, not whether the rules are the right ones (in our run the rules led to a *worse* result than a single model, and the audit passes them anyway: a correct trace is not a good decision).
- **Why 100% is easy here:** the faults are structural (changed number, flipped verdict, wrong winner, edited wording), and every claim can be recomputed. A plausible but wrong *argument* (for example a wrong rule) would not be caught.
- **For an LLM-written trace:** the same audit applies, because it works on structured claims. Ask the LLM to output claims in this format and audit them; report the share verified.
- **The 4 fault types** are ours; a real LLM could make other kinds of mistakes, so quote results as "caught X% of injected faults".

## 5. Analyst memo (assumptions and caveats), number-checked
The memo is written from a **facts dictionary** (all numbers computed above). A template writes it by default; with `GEMINI_API_KEY` an LLM may write it, and the same number-check used in your earlier notebooks rejects any number not in the facts, then falls back to the template.

In [75]:
NUM = re.compile(r"\$?\d[\d,]*\.?\d*[MK%]?")
def numbers(t): return [x.rstrip(".,") for x in NUM.findall(re.sub(r"\d{4}-\d{2}-\d{2}", " ", t))]   # numbers in a text (dates removed)

def memo_facts(s):
    model, w = select(s, cands_by_store[s], [1, 2, 3]); p = prof[s]
    return {"store": f"{s}", "n_weeks": f"{int(p['n_weeks'])}", "season": f"{p['seasonal_strength']:.2f}", "adf_p": f"{p['adf_p']:.3f}",
            "outliers": f"{100 * p['outlier_share']:.1f}%", "holiday": f"{p['holiday_effect']:.2f}", "model": model, "wmae": f"{w[model]:,.0f}",
            "sn_wmae": f"{w['SeasonalNaive']:,.0f}", "tol": f"{TOL:.0%}", "folds": "3"}

def memo_template(f):
    return "\n".join([f"**Store {f['store']} - model selection memo**",
        f"- Data: weekly sales, {f['n_weeks']} weeks, no missing weeks assumed from the profile check; seasonal strength {f['season']}, holiday weeks sell {f['holiday']}x normal weeks, {f['outliers']} of weeks are outliers.",
        f"- Decision: {f['model']} (validation WMAE {f['wmae']} vs {f['sn_wmae']} for last-year's-sales), chosen on {f['folds']} backtest folds with a {f['tol']} parsimony tolerance.",
        f"- Assumptions: the yearly pattern continues; holidays fall in the same weeks as last year; no store closures or new competitors in the next 12 weeks.",
        f"- Caveats: only {f['folds']} validation windows of 12 weeks, so the choice between close models is uncertain; stationarity test p = {f['adf_p']} is a rough guide; ETS and LightGBM settings were not tuned per store."])

def write_memo(s):
    f = memo_facts(s); text = memo_template(f); by = "template"
    if get_client() is not None:                                                            # optional LLM version, accepted only if numbers check out
        try:
            t = ask_llm("Write a 4-bullet analyst memo with sections Data, Decision, Assumptions, Caveats. Use ONLY these numbers, copied exactly: " + json.dumps(f))
            if t and all(n in set(numbers(" ".join(f.values()))) for n in numbers(t)): text, by = t, "LLM (numbers verified)"
        except Exception as e: print("LLM memo failed, using template:", type(e).__name__, "-", str(e)[:200])
    allowed = set(numbers(" ".join(f.values())) + ["3", "12"])                              # '3 folds' and '12-week' are fixed design facts
    bad = [n for n in numbers(text) if n not in allowed]
    return text, by, bad
txt, by, bad = write_memo(1)
print(txt); print(f"\n[written by {by}; numbers not in facts: {bad or 'none'}]")

* **Data**: For store 1 over 131 weeks, the Hybrid model achieved a wmae of 68,098 and a sn_wmae of 94,042 across 3 folds, with outliers detected at 14.5%.
* **Decision**: Deploy the model given the adf_p of 0.000, confirming stationarity within the required tol of 3%.
* **Assumptions**: The seasonal factor of 0.89 and the holiday multiplier of 1.08 will remain stable for future forecasting periods.
* **Caveats**: Performance may degrade if the 14.5% outlier rate increases or if store 1 experiences anomalies outside the historical 131-week baseline.

[written by LLM (numbers verified); numbers not in facts: none]


## 6. Summary for the slides
Fill the blanks from **your own run**.

In [76]:
best_single, agent = R.loc["Best single model", "mean WMAE"], R.loc["Agent (3% parsimony)", "mean WMAE"]
print(f"1. Oracle headroom: best single model is {100 * (best_single / orc - 1):.1f}% above the oracle; the agent is {100 * (agent / orc - 1):.1f}% above.")
gain = 100 * (1 - agent / best_single)
print(f"2. Agent vs best single model: mean WMAE {gain:+.1f}% ({'agent better' if gain > 0 else 'agent WORSE'}), agent better in {(a < b).mean():.0%} of stores, Wilcoxon p = {w.pvalue:.3f}.")
print(f"3. Trace audit: {100 * np.mean(ok):.0f}% of clean claims verified; {int(sum(r['caught'] for r in res))}/{int(sum(r['injected'] for r in res))} injected faults caught.")
print("4. Memo: assumptions and caveats included; numbers checked against the facts dictionary.")
summary_rows = R.reset_index().rename(columns={"index": "policy"}); summary_rows.to_csv("selector_experiment_results.csv", index=False)

1. Oracle headroom: best single model is 10.1% above the oracle; the agent is 21.2% above.
2. Agent vs best single model: mean WMAE -10.1% (agent WORSE), agent better in 2% of stores, Wilcoxon p = 0.000.
3. Trace audit: 100% of clean claims verified; 836/836 injected faults caught.
4. Memo: assumptions and caveats included; numbers checked against the facts dictionary.


## Future scope (add to the deck)
- **More and different series:** test on other public weekly datasets (for example M4 or M5 subsets) where profiles differ much more, so selection has real room to help. Needs no change to the agent.
- **More fixed policies:** a hand-written rule table ("if seasonal strength > 0.7 use ETS"), a meta-learning selector, and an AutoML baseline.
- **LLM-written traces at scale:** let the LLM write the structured claims, then measure the share verified and the types of mistakes (this notebook already provides the audit).
- **Richer candidates:** Prophet, ARIMA, TBATS, and tuned LightGBM, plus ensemble weights chosen by the agent.
- **Longer validation:** more rolling origins and several horizons, to reduce the risk of picking last period's lucky model.
- **Human evaluation:** ask analysts to rate the memos for usefulness and clarity.